# Financial Corporate Actions Alignment with OntoAligner

## From Internal Bank Vocabulary to FIBO-Standardized Mappings

Every bank speaks its own dialect. Internally, a corporate action — a dividend, a share split, a bond redemption, or an issuer call — gets a name that makes sense to the systems and people who built it:

```text
BankCashDistribution
BankShareSplitAction
BankBondMaturityRedemption
BankIssuerEarlyCallRedemption
```

These names work fine inside the bank. The trouble starts when the data has to leave the building — to a regulator, a data vendor, an API consumer, or any system that expects the industry-standard vocabulary defined by FIBO, the Financial Industry Business Ontology.

Today, that translation is often handled through hand-written mapping tables. Those mappings are brittle: when the bank changes a field name, or when an external vocabulary evolves, someone has to remember why `BankIssuerEarlyCallRedemption` is supposed to correspond to a FIBO redemption concept.

This notebook shows a different approach. Instead of rewriting the bank's schema to be “FIBO-compliant,” we use OntoAligner as a semantic bridge that helps discover mappings between the bank's vocabulary and FIBO.


![Bank to FIBO story flow](../../images/bank_fibo_story_flow.png)

## The Three Worlds That Need to Agree

Before any alignment happens, it helps to meet the three vocabularies in this story:

- **The bank's internal schema** — built for the bank's own systems, reflecting internal naming habits and shortcuts that made sense at the time they were written.
- **FIBO**, maintained by the EDM Council — the industry's attempt at a shared, rigorously defined vocabulary for financial instruments and events, including corporate actions.
- **External data providers** such as Bloomberg-style systems — which may use their own modeling conventions on top of the same underlying financial reality.

The same real-world event — say, a company calling its bonds early — gets described three different ways across these three worlds. None of them is "wrong." They simply weren't built to talk to each other.

For this demo, we focus on the first bridge: **bank vocabulary → FIBO**. Since no public bank schema exists to use directly, we built a small synthetic one that mirrors how a real bank might name its corporate-action concepts — just enough structure to make the alignment problem real, without exposing anyone's actual production schema. Bloomberg-style integration follows the same pattern, and we return to it at the end.

![Three Worlds Semantic Overlap](../../images/three_worlds_semantic_overlap.png)

## Seeing the Vocabulary Gap Before We Fix It

Before running any alignment code, it helps to look at the raw vocabulary on both sides.

The bank ontology and the FIBO ontology are describing similar corporate-action ideas, but they do not use the same naming style. This is the gap OntoAligner is meant to bridge.

The next cell prints a few class names directly from the source files, before retrieval, reranking, or postprocessing has happened.

In [1]:
# Quick look: a few concepts straight from the raw ontology files,
# before any alignment. This shows what "speaking a different dialect"
# looks like in the source and target vocabularies.

import rdflib
from rdflib import RDF, OWL


def local_name(uri):
    uri = str(uri)
    if "#" in uri:
        return uri.split("#")[-1]
    return uri.rstrip("/").split("/")[-1]


def get_class_names(file_path, limit=8):
    graph = rdflib.Graph()
    graph.parse(file_path, format="xml")

    class_names = [
        local_name(class_iri)
        for class_iri in graph.subjects(RDF.type, OWL.Class)
    ]

    return class_names[:limit]


bank_classes = get_class_names(
    "../../assets/bank-fibo/source_bank_corporate_actions.rdf",
    limit=8,
)

fibo_classes = get_class_names(
    "../../assets/bank-fibo/target_fibo_corporate_actions.rdf",
    limit=8,
)

print("Sample bank concepts:")
for class_name in bank_classes:
    print(" -", class_name)

print("\nSample FIBO concepts:")
for class_name in fibo_classes:
    print(" -", class_name)

Sample bank concepts:
 - BankTradingHaltMessage
 - BankOddLotBuybackProgram
 - BankCapitalReturnDistribution
 - BankBondMaturityRedemption
 - BankIssuerEarlyCallRedemption
 - BankIssuerRepurchaseOffer
 - BankInterestRateReset
 - BankCapitalGainPayout

Sample FIBO concepts:
 - BondDefaultAction
 - BonusIssue
 - BonusRightsIssue
 - BonusSharePlanDistribution
 - CallOnIntermediateSecurities
 - CancellationOfShares
 - CapitalDistribution
 - CapitalGainsDistribution


## Why Not Just Match the Names?

The preview above shows why this is not just a string-matching problem. Some pairs look close, such as `BankCapitalReturnDistribution` and `CapitalDistribution`, but others need more context. A term like `BankIssuerEarlyCallRedemption` may correspond to a FIBO redemption or call-related concept even if the names are not written the same way.

A tempting shortcut would be to compare class names directly. But label matching alone would miss many useful correspondences because financial concepts are often named differently across internal systems and standards.

This is why the pipeline uses OLaLa-style parsing and encoding. Instead of relying only on the class name, it pulls in richer ontology text — comments, synonyms, annotation text, normalized fragments, and related labels — and builds a fuller description to compare.

The retrieval step then uses this richer text to generate several FIBO candidates per bank concept, even when the two vocabularies do not share the exact same wording.

![Label Matching vs Rich Ontology Text](../../images/label_vs_rich_text_matching.png)

## Turning the Story into a Pipeline

Now that the problem is clear, we can turn the Bank-to-FIBO story into a working OntoAligner pipeline.

The image below gives a high-level view of the workflow. The code then implements it more concretely through loading, encoding, retrieval, reranking, postprocessing, evaluation, and export.

The important idea is that the bank keeps its own vocabulary. OntoAligner builds the bridge from that vocabulary to FIBO.

![Bank to FIBO pipeline](../../images/bank_fibo_pipeline.png)

## Setting Up the Alignment Task

We start by importing the OntoAligner components used in this notebook, then create the Bank-to-FIBO alignment task and load the source ontology, target ontology, and reference mappings.

The reference mappings are included only for evaluation later. We also select the runtime device so the retrieval and reranking models can use a GPU if available, or fall back to CPU.

In [2]:
# Import necessary libraries
import json
import torch

# Import necessary modules from the 'ontoaligner' library
# The library provides tools for ontology alignment tasks, including dataset management,
# encoding, retrieval, reranking, evaluation, and postprocessing.
from ontoaligner.encoder import OLaLaEncoder
from ontoaligner.ontology import OLaLaOMDataset
from ontoaligner.utils import metrics, xmlify
from ontoaligner.aligner import CrossEncoderReranking
from ontoaligner.aligner.olala import OLaLaSBERTRetrieval
from ontoaligner.postprocess import retriever_postprocessor

C:\Users\AlluV\Desktop\1\OntoAligner-dev-test\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Step 1: Initialize the ontology matching task
# The task is created using the OLaLa ontology dataset,
# which parses source and target ontologies while preserving richer text fields.
task = OLaLaOMDataset()

# Confirm the task initialization by printing its details
print("Bank-FIBO Task:", task)

Bank-FIBO Task: Track: Generic, Source-Target sets: OLaLa-Source-Target


In [4]:
# Step 2: Collect the ontology dataset
# The dataset includes paths to the synthetic bank source ontology,
# FIBO target ontology, and synthetic reference matchings for evaluation.
dataset = task.collect(
    source_ontology_path="../../assets/bank-fibo/source_bank_corporate_actions.rdf",
    target_ontology_path="../../assets/bank-fibo/target_fibo_corporate_actions.rdf",
    reference_matching_path="../../assets/bank-fibo/reference_matchings.xml",
)

# Confirm the dataset collection
print("Dataset Keys:", dataset.keys())
print("Number of Source Concepts:", len(dataset["source"]))
print("Number of Target Concepts:", len(dataset["target"]))
print("Number of Reference Matchings:", len(dataset["reference"]))

32it [00:00, 1391.88it/s]
54it [00:00, 1421.05it/s]
100%|██████████| 192/192 [00:00<00:00, 17456.57it/s]

Dataset Keys: dict_keys(['dataset-info', 'source', 'target', 'reference'])
Number of Source Concepts: 32
Number of Target Concepts: 53
Number of Reference Matchings: 31


In [5]:
# Step 3: Select the runtime device
# The retrieval and reranking examples can use GPU if CUDA is available;
# otherwise, they run on CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Runtime Device:", device)

Runtime Device: cpu


## Preparing the Concepts for Matching

After loading the ontologies, each concept is still an ontology concept record.

The encoder turns those records into comparable text representations. For corporate actions, this matters because useful meaning may be found not only in the label, but also in comments, annotation text, URI fragments, and normalized terms.

In [6]:
# Step 4: Initialize the encoder model
# The OLaLa encoder prepares source and target ontology concept texts
# using OLaLa TextExtractorSet fields such as labels, comments,
# annotation literals, URI fragments, and normalized text fields.
encoder_model = OLaLaEncoder()

# Encode the source and target ontologies
# The output is used as input for the retrieval model.
source_onto, target_onto = encoder_model(
    source=dataset["source"],
    target=dataset["target"],
)

print("Number of Encoded Source Concepts:", len(source_onto))
print("Number of Encoded Target Concepts:", len(target_onto))

Number of Encoded Source Concepts: 32
Number of Encoded Target Concepts: 53


## Finding Candidate FIBO Matches

The retrieval step casts a wide net.

For each bank concept, OntoAligner searches the FIBO vocabulary and returns several possible matches. At this stage, the goal is not to make the final decision yet; the goal is to make sure the correct FIBO concept is likely included in the shortlist.

In [7]:
# Step 5: Set up the retrieval model
# The OLaLa SBERT retrieval model generates candidate FIBO concepts
# for each synthetic bank concept using multiple text examples per resource.
retriever = OLaLaSBERTRetrieval(
    device=device,
    top_k=10,
    both_directions=True,
    topk_per_resource=True,
)

# Load the SBERT retrieval model
retriever.load(path="multi-qa-mpnet-base-dot-v1")

# Generate candidate alignments
# The retrieval model returns grouped candidates using source, target-cands, and score-cands.
retrieval_outputs = retriever.generate(
    input_data=[source_onto, target_onto]
)

print("Number of Retrieved Source Groups:", len(retrieval_outputs))
print("Sample Retrieval Output:", json.dumps(retrieval_outputs[:2], indent=4))

100%|██████████| 124/124 [00:00<00:00, 31018.89it/s]

Number of Retrieved Source Groups: 32
Sample Retrieval Output: [
    {
        "source": "https://example.org/bank/ontology/corporate-actions/BankTradingHaltMessage",
        "target-cands": [
            "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/TradingStatusSuspendedMessage",
            "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/TradingStatusMessage",
            "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/TradingStatusActiveMessage",
            "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/CorporateChangeOfStatusEvent",
            "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/ChangeOfSecurityTradingStatusEvent",
            "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/ListingStatus

## Reranking the Candidate Matches

Retrieval gives us a useful shortlist, but not every candidate is equally strong.

The reranker looks at each bank concept and candidate FIBO concept together, then reorders the candidates so the most semantically relevant ones rise to the top.

In [8]:
# Step 6: Set up the reranking model
# The reranker refines the candidates generated by the retrieval model.
# CrossEncoderReranking scores each source-target pair jointly and reranks the retrieved candidates.
reranker = CrossEncoderReranking(
    device=device,
    top_k=5,
    normalize_score="sigmoid",
)

# Load the CrossEncoder reranking model
reranker.load(path="cross-encoder/ms-marco-MiniLM-L6-v2")

In [9]:
# Step 7: Rerank the retrieved candidates
# The reranker preserves the retrieval output format, so the existing retriever_postprocessor
# can still be used after reranking.
reranked_outputs = reranker.generate(
    input_data=[source_onto, target_onto, retrieval_outputs]
)

print("Number of Reranked Source Groups:", len(reranked_outputs))
print("Sample Reranked Output:", json.dumps(reranked_outputs[:2], indent=4))

100%|██████████| 32/32 [00:35<00:00,  1.11s/it]

Number of Reranked Source Groups: 32
Sample Reranked Output: [
    {
        "source": "https://example.org/bank/ontology/corporate-actions/BankTradingHaltMessage",
        "target-cands": [
            "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/TradingStatusSuspendedMessage",
            "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/TradingStatusMessage",
            "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/TradingStatusActiveMessage",
            "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/ListingStatusDelistingMessage",
            "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/ChangeOfSecurityTradingStatusEvent"
        ],
        "score-cands": [
            0.9969609658989599,
            0.9316452029581145,
            0.92553210

## Converting Reranked Candidates into Matchings

The reranker returns grouped candidates for each bank concept.

The postprocessor turns those grouped candidate lists into flat source-target mappings. This is the point where reranked candidates become alignment outputs that can be evaluated, inspected, and exported.

In [10]:
# Step 8: Post-process the reranked outputs
# The retriever_postprocessor converts grouped candidates into flat source-target matchings.
# For CrossEncoderReranking with sigmoid normalization, threshold=0.5 can be used.
# For CohereReranking, threshold=0.0 keeps the reranked top-k candidates.
matchings = retriever_postprocessor(
    predicts=reranked_outputs,
    threshold=0.5,
)

print("Number of Final Matchings:", len(matchings))
print("Sample Matchings:", json.dumps(matchings[:5], indent=4))

100%|██████████| 32/32 [00:00<?, ?it/s]

Number of Final Matchings: 73
Sample Matchings: [
    {
        "source": "https://example.org/bank/ontology/corporate-actions/BankTradingHaltMessage",
        "target": "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/TradingStatusSuspendedMessage",
        "score": 0.9969609658989599
    },
    {
        "source": "https://example.org/bank/ontology/corporate-actions/BankTradingHaltMessage",
        "target": "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/TradingStatusMessage",
        "score": 0.9316452029581145
    },
    {
        "source": "https://example.org/bank/ontology/corporate-actions/BankTradingHaltMessage",
        "target": "https://spec.edmcouncil.org/fibo/ontology/CAE/CorporateEvents/SecurityRelatedCorporateActions/TradingStatusActiveMessage",
        "score": 0.9255321028165582
    },
    {
        "source": "https://example.org/bank/ontology/corporate-actions/BankOddLotBuyba

## Two Different Questions, Two Different Answers

“Show me everything that might match” and “tell me the one strongest answer” are not the same request, and a useful alignment pipeline should be able to support both.

In the first mode, the postprocessing step keeps every candidate above the reranker’s confidence threshold. This is useful when a data steward or domain expert wants to review several plausible FIBO matches for a bank concept before committing to one.

In the second mode, the strict top-1 pass keeps only the single best candidate per concept. This is the version you would more likely wire into an automated workflow or expose through an API portal as a final equivalence mapping.

Neither mode is “more correct.” They answer two different real-world needs: exploration for review, and production for downstream use.

![Candidate Discovery vs Final Alignment](../../images/candidate_vs_final_alignment.png)

## Candidate Discovery Mode

We first keep all reranked candidates above the confidence threshold.

This creates a review-friendly set of possible Bank-to-FIBO mappings before choosing a single final match.

In [11]:
# Step 9: Evaluate the generated matchings
# The evaluation report compares predicted matchings against the synthetic reference alignments
# using metrics such as precision, recall, and F-score.
evaluation = metrics.evaluation_report(
    predicts=matchings,
    references=dataset["reference"],
)

# Print the evaluation report in a human-readable JSON format
print("Evaluation Report:", json.dumps(evaluation, indent=4))

Evaluation Report: {
    "intersection": 30,
    "precision": 41.0958904109589,
    "recall": 96.7741935483871,
    "f-score": 57.692307692307686,
    "predictions-len": 73,
    "reference-len": 31
}


## Strict Top-1 Final Alignment

Next, we keep only the highest-ranked FIBO candidate for each bank concept.

This produces a cleaner one-to-one alignment output for downstream use, such as an API portal or metadata layer.

In [12]:
# Create a stricter top-1 version of the reranked outputs
# This keeps only the highest-ranked FIBO candidate for each source concept.
strict_reranked_outputs = []

for output in reranked_outputs:
    if not output["target-cands"]:
        continue

    strict_reranked_outputs.append({
        "source": output["source"],
        "target-cands": output["target-cands"][:1],
        "score-cands": output["score-cands"][:1],
    })

strict_matchings = retriever_postprocessor(
    predicts=strict_reranked_outputs,
    threshold=0.5,
)

strict_evaluation = metrics.evaluation_report(
    predicts=strict_matchings,
    references=dataset["reference"],
)

print("Strict Top-1 Evaluation Report:", json.dumps(strict_evaluation, indent=4))

100%|██████████| 32/32 [00:00<00:00, 31895.85it/s]

Strict Top-1 Evaluation Report: {
    "intersection": 27,
    "precision": 87.09677419354838,
    "recall": 87.09677419354838,
    "f-score": 87.09677419354838,
    "predictions-len": 31,
    "reference-len": 31
}


## Exporting the Alignment Results

Once the mappings are generated, they need to be usable outside the notebook.

The JSON outputs are useful for inspection, debugging, and application logic. The XML alignment outputs are useful for ontology-alignment workflows that expect a standard alignment file.

In [13]:
# Step 10: Export matchings in XML format or JSON format

# XML format
# Convert the generated matchings into an XML alignment file using the xmlify utility.
xml_str = xmlify.xml_alignment_generator(matchings=matchings)

# Save the XML alignment to a file for further use or analysis
output_file_path = "bank_fibo_reranked_matchings.xml"
with open(output_file_path, "w", encoding="utf-8") as xml_file:
    xml_file.write(xml_str)

print(f"Matchings in XML format have been successfully written to '{output_file_path}'.")

# JSON format
# Save the generated matchings in dictionary format for further analysis or debugging.
output_file_path = "bank_fibo_reranked_matchings.json"
with open(output_file_path, "w", encoding="utf-8") as json_file:
    json.dump(matchings, json_file, indent=4, ensure_ascii=False)

print(f"Matchings in JSON format have been successfully written to '{output_file_path}'.")

Matchings in XML format have been successfully written to 'bank_fibo_reranked_matchings.xml'.
Matchings in JSON format have been successfully written to 'bank_fibo_reranked_matchings.json'.


In [14]:
# Export strict top-1 matchings as JSON
strict_output_json_path = "bank_fibo_strict_top1_matchings.json"

with open(strict_output_json_path, "w", encoding="utf-8") as json_file:
    json.dump(strict_matchings, json_file, indent=4, ensure_ascii=False)

print(f"Strict top-1 matchings in JSON format have been written to '{strict_output_json_path}'.")


# Export strict top-1 matchings as XML
strict_output_xml_path = "bank_fibo_strict_top1_matchings.xml"

strict_xml_str = xmlify.xml_alignment_generator(matchings=strict_matchings)

with open(strict_output_xml_path, "w", encoding="utf-8") as xml_file:
    xml_file.write(strict_xml_str)

print(f"Strict top-1 matchings in XML format have been written to '{strict_output_xml_path}'.")

Strict top-1 matchings in JSON format have been written to 'bank_fibo_strict_top1_matchings.json'.
Strict top-1 matchings in XML format have been written to 'bank_fibo_strict_top1_matchings.xml'.


## What This Demo Actually Showed

Using a synthetic bank vocabulary standing in for a real internal schema, and a FIBO corporate-actions subset as the target, the pipeline — OLaLa parsing, OLaLa encoding, SBERT-based retrieval, and cross-encoder reranking — produced equivalence mappings without any hand-written rules.

The results from this run are encouraging. In candidate discovery mode, the pipeline recovered **30 of 31** intended mappings, meaning the correct FIBO concept was almost always present somewhere in the retrieved candidate set. Because this mode intentionally keeps multiple candidates for review, its F-score was **57.69%**.

In strict top-1 mode — the version more suitable for automated, production-style use — the pipeline recovered **27 of 31** mappings, with an F-score of **87.10%**.

This notebook uses synthetic data only to make the demo concrete, shareable, and safe to publish. The workflow itself is the important part: the same steps apply unchanged to a real internal bank schema once that vocabulary is represented as an ontology or schema-derived vocabulary.

The same workflow also extends naturally to the third world in this story: a Bloomberg-style external provider model can be treated as another source ontology and aligned to FIBO in exactly the same way. In that setup, FIBO becomes the shared meeting point — the bank and the provider do not need to agree with each other directly, only with the common standard sitting between them.